# LangChain Fundamentals: Part 2
## Messages and Conversational AI

### Learning Objectives
By the end of this notebook, you'll understand:
1. **Message Types** - SystemMessage, HumanMessage, AIMessage
2. **Conversations** - How to maintain context across multiple turns
3. **Chat Prompt Templates** - Creating structured prompts for conversations
4. **Message History** - Appending new messages to maintain state
5. **Real-world Applications** - Building chatbots and conversational agents

## Setup: Environment and Models

Same as Part 1, we'll load our environment and initialize the models:

In [1]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

chat = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("✓ Environment and models ready!")

✓ Environment and models ready!


## What Are Messages?

In LangChain, a **message** is a structured way to represent a conversation turn.

### Why Messages?
- **Explicit** - We know who said what (system, human, or AI)
- **Ordered** - Messages maintain conversation flow
- **Traceable** - We can see the entire conversation history
- **Flexible** - Different message types for different roles

### Three Main Message Types:

| Type | Purpose | Example |
|------|---------|----------|
| **SystemMessage** | Set the AI's behavior/personality | "You are a helpful assistant" |
| **HumanMessage** | User's input | "Explain string theory" |
| **AIMessage** | AI's response | "String theory is..." |

## Step 1: Understanding Message Types

Let's import the message types and see what they look like:

In [2]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# SystemMessage - tells the AI how to behave
system_msg = SystemMessage(content="You are a helpful physics teacher.")

# HumanMessage - user asks a question
human_msg = HumanMessage(content="Hi AI, how are you today?")

# AIMessage - AI responds
ai_msg = AIMessage(content="I'm great thank you. How can I help you?")

print("SystemMessage:")
print(f"  Type: {system_msg.type}")
print(f"  Content: {system_msg.content}")

print("\nHumanMessage:")
print(f"  Type: {human_msg.type}")
print(f"  Content: {human_msg.content}")

print("\nAIMessage:")
print(f"  Type: {ai_msg.type}")
print(f"  Content: {ai_msg.content}")

SystemMessage:
  Type: system
  Content: You are a helpful physics teacher.

HumanMessage:
  Type: human
  Content: Hi AI, how are you today?

AIMessage:
  Type: ai
  Content: I'm great thank you. How can I help you?


## Step 2: Building a Conversation with Messages

A conversation is just a list of messages in order. Let's build one about string theory:

In [3]:
# Build a conversation message by message
messages = [
    SystemMessage(content="You are a helpful physics teacher who explains complex concepts simply."),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

print("Conversation so far:")
print("=" * 60)
for i, msg in enumerate(messages, 1):
    print(f"\n{i}. [{msg.type.upper()}]")
    print(f"   {msg.content[:100]}..." if len(msg.content) > 100 else f"   {msg.content}")

Conversation so far:

1. [SYSTEM]
   You are a helpful physics teacher who explains complex concepts simply.

2. [HUMAN]
   Hi AI, how are you today?

3. [AI]
   I'm great thank you. How can I help you?

4. [HUMAN]
   I'd like to understand string theory.


## Step 3: Getting the AI's Response

Now we'll send all these messages to the chat model and get a response:

In [4]:
# Send the conversation to the chat model
response = chat.invoke(messages)

print("AI Response:")
print("=" * 60)
print(response.content)

AI Response:
String theory is a theoretical framework in which the point-like particles of particle physics are replaced by one-dimensional objects called "strings." These strings can vibrate at different frequencies, and their different vibrational modes are thought to correspond to the various fundamental particles, like electrons and quarks.

Here's a simple breakdown of the key ideas:

1. **Strings, Not Particles**: In traditional particle physics, particles are considered point-like with no dimensions. In string theory, these particles are actually tiny, vibrating strings. The type of particle is determined by the string's vibration pattern.

2. **Dimensions**: String theory requires more than the usual four dimensions (three of space and one of time). The most common version, called superstring theory, suggests there are 10 dimensions. These extra dimensions are thought to be compactified or curled up so small that we don't notice them in everyday life.

3. **Unification**: One o

## Step 4: Continuing the Conversation

To keep the conversation going, we:
1. Add the AI's response to our messages
2. Add the human's next question
3. Call the model again

This maintains **context** - the AI remembers what was said before.

In [5]:
# Add the AI's response to our messages
messages.append(response)

# Add a follow-up question
follow_up = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)
messages.append(follow_up)

print("Updated conversation:")
print(f"Total messages: {len(messages)}")
print("\nLast two messages:")
for msg in messages[-2:]:
    print(f"\n[{msg.type.upper()}]")
    print(msg.content[:150] + "..." if len(msg.content) > 150 else msg.content)

Updated conversation:
Total messages: 6

Last two messages:

[AI]
String theory is a theoretical framework in which the point-like particles of particle physics are replaced by one-dimensional objects called "strings...

[HUMAN]
Why do physicists believe it can produce a 'unified theory'?


## Step 5: Get the AI's Follow-up Response

Send all messages again. The AI will consider the full conversation history:

In [6]:
# Call the model with the full conversation
response = chat.invoke(messages)

print("AI's Follow-up Response:")
print("=" * 60)
print(response.content)

AI's Follow-up Response:
Physicists are interested in string theory as a potential "unified theory" because it offers a framework that could, in principle, combine all the fundamental forces of nature into a single, coherent theoretical structure. Here's why string theory is seen as a promising candidate for unification:

1. **Inclusion of Gravity**: One of the biggest challenges in theoretical physics is reconciling general relativity, which describes gravity, with quantum mechanics, which describes the other three fundamental forces (electromagnetic, weak, and strong nuclear forces). String theory naturally incorporates gravity, as one of the vibrational modes of a string corresponds to the graviton, the hypothetical quantum particle that mediates the gravitational force.

2. **Quantum Consistency**: String theory is formulated in a way that is consistent with the principles of quantum mechanics. This is crucial because any unified theory must work at both the large scales of the cos

## How Message History Works

### Key Insight:
Every time you call `chat.invoke(messages)`, you send **all previous messages**.

```python
# First call
response1 = chat.invoke([system, human1])

# Second call - includes first message + new one
response2 = chat.invoke([system, human1, ai1, human2])
```

### The Pattern:
1. Start with system message(s)
2. Add first human message
3. Get AI response
4. **Keep ALL previous messages**
5. Add next human message
6. Get AI response again
7. Repeat...

## Step 6: System Messages - Controlling Behavior

The system message is crucial - it defines the AI's personality and instructions.

Let's see how different system messages change the AI's behavior:

In [7]:
# Example: Expert trader persona
trader_messages = [
    SystemMessage(content=(
        "You are an expert financial analyst specializing in commodities and safe-haven assets. "
        "You provide clear, practical investment insights based on market fundamentals. "
        "Keep responses concise and actionable."
    )),
    HumanMessage(content="What are safe-haven assets and why do investors buy them?")
]

response = chat.invoke(trader_messages)

print("Expert Trader Response:")
print("=" * 60)
print(response.content)

Expert Trader Response:
Safe-haven assets are investments that are expected to retain or increase in value during times of market turbulence or economic uncertainty. Investors buy them to protect their portfolios from volatility and potential losses. Common safe-haven assets include:

1. **Gold**: Traditionally seen as a store of value, gold is often sought after during inflationary periods or geopolitical tensions.
2. **U.S. Treasuries**: Considered one of the safest investments due to the backing of the U.S. government, they provide a reliable income stream.
3. **Swiss Franc (CHF)**: Known for its stability, the Swiss Franc is often favored during financial crises.
4. **Japanese Yen (JPY)**: Similar to the Swiss Franc, the Yen is considered a stable currency in times of global uncertainty.
5. **Defensive Stocks**: Companies in sectors like utilities, healthcare, and consumer staples tend to perform well regardless of economic conditions.

Investors buy these assets to diversify their

## Step 7: Chat Prompt Templates

Just like in Part 1, we can create templates for chat messages.
This makes it easy to build conversations dynamically.

### Why Chat Prompt Templates?
- Create reusable conversation structures
- Mix different message types (system, human, AI)
- Add variables for dynamic content

In [8]:
from langchain_core.prompts import (
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)

# Create a template for a human message with a constraint
human_template = HumanMessagePromptTemplate.from_template(
    '{input} '
    'Keep your response to no more than 100 characters (including spaces).'
)

# Create a chat prompt from just the human template
chat_prompt = ChatPromptTemplate.from_messages([human_template])

# Format it with actual input
chat_prompt_value = chat_prompt.format_prompt(
    input="What are safe-haven assets?"
)

print("Formatted chat prompt:")
print(chat_prompt_value)

print("\nConverted to messages:")
print(chat_prompt_value.to_messages())

Formatted chat prompt:
messages=[HumanMessage(content='What are safe-haven assets? Keep your response to no more than 100 characters (including spaces).', additional_kwargs={}, response_metadata={})]

Converted to messages:
[HumanMessage(content='What are safe-haven assets? Keep your response to no more than 100 characters (including spaces).', additional_kwargs={}, response_metadata={})]


## Step 8: Building Complex Chat Prompt Templates

Now let's create a template with multiple message types:

In [9]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate
)

# Create a system message template
system_template = SystemMessagePromptTemplate.from_template(
    'You are a helpful assistant that keeps responses to no more than '
    '{character_limit} characters (including whitespace). '
    'Always sign off with "- {sign_off}"'
)

# Create templates for human and AI messages
human_template = HumanMessagePromptTemplate.from_template("{input}")
ai_template = AIMessagePromptTemplate.from_template("{response} - {sign_off}")

# Combine them into a chat prompt template
chat_prompt = ChatPromptTemplate.from_messages([
    system_template,
    human_template,
    ai_template
])

# Format with actual values
chat_prompt_value = chat_prompt.format_prompt(
    character_limit="80",
    sign_off="Finance Bot",
    input="What are safe-haven assets?",
    response="Assets that retain value during market downturns and economic uncertainty."
)

print("Complex chat prompt with 3 message types:")
print(chat_prompt_value)

print("\n" + "="*60)
print("\nAs messages:")
for msg in chat_prompt_value.to_messages():
    print(f"\n[{msg.type.upper()}]")
    print(msg.content)

Complex chat prompt with 3 message types:
messages=[SystemMessage(content='You are a helpful assistant that keeps responses to no more than 80 characters (including whitespace). Always sign off with "- Finance Bot"', additional_kwargs={}, response_metadata={}), HumanMessage(content='What are safe-haven assets?', additional_kwargs={}, response_metadata={}), AIMessage(content='Assets that retain value during market downturns and economic uncertainty. - Finance Bot', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


As messages:

[SYSTEM]
You are a helpful assistant that keeps responses to no more than 80 characters (including whitespace). Always sign off with "- Finance Bot"

[HUMAN]
What are safe-haven assets?

[AI]
Assets that retain value during market downturns and economic uncertainty. - Finance Bot


## Step 9: Using Chat Prompts with the Model

Now let's use our chat prompt template with the actual model:

In [10]:
# Get the messages from our template
messages = chat_prompt_value.to_messages()

# Add a follow-up question
follow_up = HumanMessage(
    content="Is agricultural commodities a safe-haven asset?"
)
messages.append(follow_up)

print("Messages sent to model:")
for i, msg in enumerate(messages, 1):
    print(f"{i}. [{msg.type.upper()}] {msg.content[:60]}...")

print("\n" + "="*60)
print("\nAI Response:")

response = chat.invoke(messages)
print(response.content)

Messages sent to model:
1. [SYSTEM] You are a helpful assistant that keeps responses to no more ...
2. [HUMAN] What are safe-haven assets?...
3. [AI] Assets that retain value during market downturns and economi...
4. [HUMAN] Is agricultural commodities a safe-haven asset?...


AI Response:
Generally, agricultural commodities are not considered safe-haven assets. - Finance Bot


## Step 10: Practical Example - Building a Simple Chatbot

Let's put it all together and build a simple chatbot:

In [11]:
# Initialize conversation
conversation_history = [
    SystemMessage(content=(
        "You are a knowledgeable but friendly AI tutor. "
        "Explain concepts clearly without being condescending. "
        "Keep responses concise and ask follow-up questions when helpful."
    ))
]

# Simulate a multi-turn conversation
user_inputs = [
    "What is machine learning?",
    "How is it different from deep learning?",
    "Can you give a simple example?"
]

print("=" * 70)
print("CHATBOT CONVERSATION")
print("=" * 70)

for i, user_input in enumerate(user_inputs, 1):
    print(f"\nTurn {i}:")
    print(f"User: {user_input}")
    
    # Add user message
    conversation_history.append(HumanMessage(content=user_input))
    
    # Get AI response
    response = chat.invoke(conversation_history)
    
    # Add AI response to history
    conversation_history.append(response)
    
    # Display response
    print(f"Bot: {response.content[:200]}..." if len(response.content) > 200 else f"Bot: {response.content}")

print("\n" + "="*70)
print(f"\nConversation length: {len(conversation_history)} messages")
print(f"(1 system + {len(user_inputs)} user inputs + {len(user_inputs)} AI responses)")

CHATBOT CONVERSATION

Turn 1:
User: What is machine learning?
Bot: Machine learning is a branch of artificial intelligence that focuses on building systems that can learn from and make decisions based on data. Instead of being explicitly programmed to perform a task,...

Turn 2:
User: How is it different from deep learning?
Bot: Deep learning is actually a subset of machine learning. While machine learning encompasses a wide range of algorithms and techniques, deep learning specifically refers to models that use neural networ...

Turn 3:
User: Can you give a simple example?
Bot: Sure! Let's consider a simple example of image classification using deep learning.

Imagine you want to build a system that can automatically identify whether an image contains a cat or a dog. Here's ...


Conversation length: 7 messages
(1 system + 3 user inputs + 3 AI responses)


## Understanding Token Costs

**Important:** Each API call sends **all previous messages**.

```python
Turn 1: Send [system, user1] → Cost: 2 messages
Turn 2: Send [system, user1, ai1, user2] → Cost: 4 messages
Turn 3: Send [system, user1, ai1, user2, ai2, user3] → Cost: 6 messages
```

**Solution:** For long conversations, consider:
- Using summarization to condense old messages
- Implementing a sliding window (keeping only recent messages)
- Using memory systems (covered in later notebooks)

## Summary: What We Learned

### Key Concepts:
1. **Message Types** - SystemMessage, HumanMessage, AIMessage
2. **Conversation Flow** - Ordered list of messages
3. **Context Preservation** - Send all previous messages to maintain context
4. **System Messages** - Control AI behavior and personality
5. **Chat Prompt Templates** - Build reusable conversation structures
6. **Multi-turn Conversations** - Build interactive chatbots

### The Message Pattern:
```python
# Start conversation
messages = [SystemMessage(...)]

# Each turn
messages.append(HumanMessage(input))
response = chat.invoke(messages)
messages.append(response)
```

### Next Steps:
- In **Part 3**, we'll learn about **memory systems** and LangGraph
- How to build more sophisticated agents
- How to integrate external tools and data sources

## Practice Exercises

### Exercise 1: Modify System Message
Change the tutor's system message to be a "cynical code reviewer" and see how responses differ.

### Exercise 2: Build a Specialized Chatbot
Create a chatbot for a specific domain (chef, historian, comedian, etc.) and have a 3-turn conversation.

### Exercise 3: Message Templates
Create a chat prompt template that includes system, human, and AI messages with your own variables.

### Exercise 4: Analyze Token Usage
Print out `response.response_metadata['token_usage']` to see how tokens increase with each turn.